In [47]:
import numpy as np
import scipy
from cascadoc_old import Solver, Mesh
from cascadoc_old.problem import HypProblem
import time

In [48]:
class NewSolver(Solver):
    pass

In [49]:
x0 = lambda s: s**2
y0 = lambda s: 1/2

T = [0, 0.5]
S = [0, 1]
C = [1,2]

B11 = lambda s, t: 3 * s * t
B12 = lambda s, t: np.sin(t)
B21 = lambda s, t: np.cos(s)
B22 = lambda s, t: 2*s+t

G11 = lambda t: t**2
G12 = lambda t: 0

G21 = lambda t: 0
G22 = lambda t: t/2

F1 = lambda s, t: 4*np.exp(s-t)
F2 = lambda s, t: 3*np.exp(t-s)

phi_dx = None
phi_dy = None 

hyp_problem = HypProblem(T=T, S=S, C=C, B=[[B11, B12], [B21, B22]], 
                         F=[F1, F2], G=[[G11, G12], [G21, G22]], 
                         X0=x0, Y0=y0, phi_dx=phi_dx, phi_dy=phi_dy)

mesh = Mesh(hyp_problem, 100)
solver = Solver()
solver.solve_initial(mesh, hyp_problem)
solver.solver_center(mesh, hyp_problem)
solver.solver_final(mesh, hyp_problem)

In [50]:
nodes = mesh.get_border(type_border="left", sort_t=True)
t_mesh = [node[0][3] for node in nodes]
U0 = [G22(ti) for ti in t_mesh]
def J(U):
    G22_0 = lambda t: np.interp(t, t_mesh, U)
    G21_0 = hyp_problem.G21
    G11_0 = hyp_problem.G11
    G12_0 = hyp_problem.G12
    
    hyp_problem.set_G([[G11_0, G12_0], [G21_0, G22_0]])
    solver.solve_initial(mesh, hyp_problem)
    solver.solver_center(mesh, hyp_problem)
    solver.solver_final(mesh, hyp_problem)
    nodes = mesh.get_border(type_border="final", sort_s=True)
    s_final = [node[0][2] for i, node in enumerate(nodes)]
    x_final = [node[1][0][0] for i, node in enumerate(nodes)]
    y_final = [node[1][0][1] for i, node in enumerate(nodes)]

    f = [xi+yi for xi, yi, si in zip(x_final, y_final, s_final)]
    # for fi in f:
    #     print(f'{fi:.2f} ', end='')
    # print()
    return scipy.integrate.trapezoid(f, s_final)

rez = J(U0)
print(f'J(U0): {rez:.4f}')
st = time.time()
rez = scipy.optimize.minimize(J, U0, method="BFGS", tol=0.001)
rez = J( rez.x)
fi = time.time()
print(f'time: {fi-st:.2f} sec.')
print(f'J(U*): {rez:.4f}')


J(U0): 5.9836


IOStream.flush timed out


time: 99.20 sec.
J(U*): 5.3834
